# Dataset loading

This section loads the local Polish road sign classification dataset and checks the number of images in each class.

In [ ]:
from pathlib import Path
import math
import random
import shutil

import pandas as pd
import matplotlib.pyplot as plt


In [ ]:
PROJECT_ROOT = next(
    (
        path for path in [Path.cwd(), *Path.cwd().parents]
        if (path / "data" / "classification").exists()
    ),
    None,
)

if PROJECT_ROOT is None:
    raise FileNotFoundError(
        "Could not find data/classification. "
        "Open this notebook from the project folder or from notebooks/."
    )

DATA_DIR = PROJECT_ROOT / "data"
CLASSIFICATION_DIR = DATA_DIR / "classification"

CLASSIFICATION_DIR.exists()


In [ ]:
class_dirs = sorted([path for path in CLASSIFICATION_DIR.iterdir() if path.is_dir()])
class_names = [path.name for path in class_dirs]

class_names


In [ ]:
image_extensions = {".jpg", ".jpeg", ".png"}

class_counts = []

for class_dir in class_dirs:
    image_count = len([
        path for path in class_dir.rglob("*")
        if path.suffix.lower() in image_extensions
    ])

    class_counts.append({
        "class_name": class_dir.name,
        "image_count": image_count,
    })

df_classes = pd.DataFrame(class_counts)
df_classes


In [ ]:
total_images = df_classes["image_count"].sum()
total_images


# Exploratory data analysis

This section checks the basic structure of the classification dataset.

In [ ]:
num_classes = len(df_classes)
num_classes


In [ ]:
df_classes.sort_values("image_count").head()


In [ ]:
df_classes.sort_values("image_count", ascending=False).head()


In [ ]:
plt.figure(figsize=(12, 6))
plt.bar(df_classes["class_name"], df_classes["image_count"])
plt.xticks(rotation=90)
plt.xlabel("Class")
plt.ylabel("Number of images")
plt.title("Number of images per class")
plt.show()


# Sample road sign images

This section displays example images from selected road sign classes.

In [ ]:
sample_images = []

for class_dir in class_dirs:
    image_paths = sorted([
        path for path in class_dir.rglob("*")
        if path.suffix.lower() in image_extensions
    ])

    if image_paths:
        sample_images.append({
            "class_name": class_dir.name,
            "image_path": image_paths[0],
        })

len(sample_images)


In [ ]:
columns = 4
rows = math.ceil(len(sample_images) / columns)

fig, axes = plt.subplots(rows, columns, figsize=(12, rows * 3))
axes = axes.flatten()

for ax, sample in zip(axes, sample_images):
    image = plt.imread(sample["image_path"])
    ax.imshow(image)
    ax.set_title(sample["class_name"])
    ax.axis("off")

for ax in axes[len(sample_images):]:
    ax.axis("off")

plt.tight_layout()
plt.show()


# Data preprocessing

This section prepares the classification dataset split for YOLO11 training.

In [ ]:
PROCESSED_DATA_DIR = DATA_DIR / "processed" / "yolo11_classification"

TRAIN_RATIO = 0.70
VAL_RATIO = 0.15
TEST_RATIO = 0.15
RANDOM_SEED = 42

TRAIN_RATIO + VAL_RATIO + TEST_RATIO


In [ ]:
image_records = []

for class_dir in class_dirs:
    image_paths = sorted([
        path for path in class_dir.rglob("*")
        if path.suffix.lower() in image_extensions
    ])

    for image_path in image_paths:
        image_records.append({
            "class_name": class_dir.name,
            "image_path": image_path,
        })

df_images = pd.DataFrame(image_records)
df_images.head()


In [ ]:
rng = random.Random(RANDOM_SEED)
split_records = []

for class_name, group in df_images.groupby("class_name"):
    image_paths = group["image_path"].tolist()
    rng.shuffle(image_paths)

    total_count = len(image_paths)
    train_count = int(total_count * TRAIN_RATIO)
    val_count = int(total_count * VAL_RATIO)

    train_paths = image_paths[:train_count]
    val_paths = image_paths[train_count:train_count + val_count]
    test_paths = image_paths[train_count + val_count:]

    for split_name, paths in [
        ("train", train_paths),
        ("val", val_paths),
        ("test", test_paths),
    ]:
        for image_path in paths:
            split_records.append({
                "split": split_name,
                "class_name": class_name,
                "image_path": image_path,
            })

df_split = pd.DataFrame(split_records)
df_split.head()


In [ ]:
split_summary = (
    df_split
    .groupby(["split", "class_name"])
    .size()
    .reset_index(name="image_count")
)

split_summary.head()


In [ ]:
df_split["split"].value_counts()


# Przygotowanie danych dla YOLO11

Ta cz??? tworzy struktur? katalog?w wymagan? przez klasyfikacj? YOLO: `train`, `val` i `test`. Ka?dy z tych katalog?w zawiera podfoldery z nazwami klas.


In [ ]:
def prepare_yolo11_classification_dataset(df_split, output_dir, overwrite=False):
    output_dir = Path(output_dir)

    if overwrite and output_dir.exists():
        shutil.rmtree(output_dir)

    output_dir.mkdir(parents=True, exist_ok=True)

    for row in df_split.itertuples(index=False):
        destination_dir = output_dir / row.split / row.class_name
        destination_dir.mkdir(parents=True, exist_ok=True)

        destination_path = destination_dir / row.image_path.name
        shutil.copy2(row.image_path, destination_path)

    return output_dir


In [ ]:
PREPARE_DATASET = True
OVERWRITE_DATASET = False

if PREPARE_DATASET:
    prepare_yolo11_classification_dataset(
        df_split=df_split,
        output_dir=PROCESSED_DATA_DIR,
        overwrite=OVERWRITE_DATASET,
    )

required_split_dirs = [
    PROCESSED_DATA_DIR / "train",
    PROCESSED_DATA_DIR / "val",
    PROCESSED_DATA_DIR / "test",
]

[path.exists() for path in required_split_dirs]


In [ ]:
REPORTS_DIR = PROJECT_ROOT / "reports"
REPORTS_DIR.mkdir(parents=True, exist_ok=True)

split_summary_path = REPORTS_DIR / "dataset_split_summary.csv"
split_summary.to_csv(split_summary_path, index=False)

split_summary_path


# Trening YOLO11

Domy?lnie trenowany jest tylko `yolo11n-cls.pt`, bo jest najmniejszy i najprostszy do uruchomienia. Do por?wnania modeli mo?na ustawi? `COMPARE_MODELS = True`, wtedy notebook uruchomi te? `yolo11s-cls.pt` i `yolo11m-cls.pt`.


In [ ]:
try:
    from ultralytics import YOLO
except ModuleNotFoundError as exc:
    if exc.name == "ultralytics":
        raise ModuleNotFoundError(
            "Brakuje biblioteki ultralytics w aktywnym kernelu. "
            "Uruchom w notebooku: %pip install -r ../requirements.txt"
        ) from exc
    raise


In [ ]:
YOLO_DATA_DIR = PROCESSED_DATA_DIR
TRAINING_RUNS_DIR = PROJECT_ROOT / "runs" / "yolo11_classification"
MODELS_DIR = PROJECT_ROOT / "models"
MODELS_DIR.mkdir(parents=True, exist_ok=True)

COMPARE_MODELS = False
YOLO_MODELS = ["yolo11n-cls.pt"]

if COMPARE_MODELS:
    YOLO_MODELS = [
        "yolo11n-cls.pt",
        "yolo11s-cls.pt",
        "yolo11m-cls.pt",
    ]

IMAGE_SIZE = 224
EPOCHS = 25
BATCH_SIZE = 32
DEVICE = None  # ustaw "cpu", je?li chcesz wymusi? CPU, albo 0 dla GPU

YOLO_MODELS


In [ ]:
RUN_TRAINING = False

training_results = {}

if RUN_TRAINING:
    for model_name in YOLO_MODELS:
        model = YOLO(model_name)
        run_name = model_name.replace("-cls.pt", "")

        train_kwargs = {
            "data": str(YOLO_DATA_DIR),
            "epochs": EPOCHS,
            "imgsz": IMAGE_SIZE,
            "batch": BATCH_SIZE,
            "project": str(TRAINING_RUNS_DIR),
            "name": run_name,
            "exist_ok": True,
        }
        if DEVICE is not None:
            train_kwargs["device"] = DEVICE

        results = model.train(**train_kwargs)
        training_results[model_name] = results

training_results


# Por?wnanie wynik?w i zapis modelu

Po treningu Ultralytics zapisuje wyniki w `runs/yolo11_classification`. Poni?sze kom?rki zbieraj? ko?cowe metryki z `results.csv` i kopiuj? najlepsze wagi do katalogu `models`.


In [ ]:
def read_last_metrics(run_dir):
    results_csv = run_dir / "results.csv"
    if not results_csv.exists():
        return None

    metrics = pd.read_csv(results_csv)
    if metrics.empty:
        return None

    last_row = metrics.tail(1).copy()
    last_row.insert(0, "run", run_dir.name)
    return last_row

metrics_frames = []

for model_name in YOLO_MODELS:
    run_name = model_name.replace("-cls.pt", "")
    run_dir = TRAINING_RUNS_DIR / run_name
    last_metrics = read_last_metrics(run_dir)

    if last_metrics is not None:
        metrics_frames.append(last_metrics)

if metrics_frames:
    model_comparison = pd.concat(metrics_frames, ignore_index=True)
    model_comparison.to_csv(REPORTS_DIR / "training_summary.csv", index=False)
else:
    model_comparison = pd.DataFrame()

model_comparison


In [ ]:
best_model_paths = []

for model_name in YOLO_MODELS:
    run_name = model_name.replace("-cls.pt", "")
    best_model_path = TRAINING_RUNS_DIR / run_name / "weights" / "best.pt"

    if best_model_path.exists():
        saved_model_path = MODELS_DIR / f"{run_name}-best.pt"
        shutil.copy2(best_model_path, saved_model_path)
        best_model_paths.append(saved_model_path)

best_model_paths


# Walidacja i macierz pomy?ek

Ta cz??? uruchamia walidacj? najlepszego modelu. Ultralytics zapisuje wykresy i macierz pomy?ek w katalogu `runs`.


In [ ]:
RUN_VALIDATION = False
MODEL_TO_VALIDATE = MODELS_DIR / "yolo11n-best.pt"

validation_results = None

if RUN_VALIDATION:
    if not MODEL_TO_VALIDATE.exists():
        raise FileNotFoundError(f"Nie znaleziono modelu: {MODEL_TO_VALIDATE}")

    validation_model = YOLO(str(MODEL_TO_VALIDATE))
    validation_results = validation_model.val(
        data=str(YOLO_DATA_DIR),
        imgsz=IMAGE_SIZE,
        project=str(TRAINING_RUNS_DIR),
        name="validation",
        plots=True,
    )

validation_results


# Wizualne por?wnanie modeli

Ta sekcja zbiera wyniki zapisane przez YOLO11 i pokazuje je w prostych tabelach oraz wykresach. Je?eli wytrenowany jest tylko `YOLO11n`, notebook poka?e wynik dla jednego modelu. Po ustawieniu `COMPARE_MODELS = True` i treningu `YOLO11s` oraz `YOLO11m` te same kom?rki automatycznie poka?? por?wnanie kilku modeli.


In [ ]:
def model_label_from_run(run_name):
    labels = {
        "yolo11n": "YOLO11n",
        "yolo11s": "YOLO11s",
        "yolo11m": "YOLO11m",
    }
    return labels.get(run_name, run_name)


def find_trained_models():
    model_paths = {}

    for model_path in sorted(MODELS_DIR.glob("*-best.pt")):
        run_name = model_path.stem.replace("-best", "")
        model_paths[run_name] = model_path

    if TRAINING_RUNS_DIR.exists():
        for weights_path in sorted(TRAINING_RUNS_DIR.glob("*/weights/best.pt")):
            run_name = weights_path.parents[1].name
            model_paths.setdefault(run_name, weights_path)

    return model_paths


trained_model_paths = find_trained_models()
trained_model_paths


In [ ]:
def collect_training_metrics(runs_dir):
    rows = []

    if not runs_dir.exists():
        return pd.DataFrame()

    for results_csv in sorted(runs_dir.glob("*/results.csv")):
        run_name = results_csv.parent.name
        metrics = pd.read_csv(results_csv)

        if metrics.empty:
            continue

        last_row = metrics.tail(1).iloc[0].to_dict()
        last_row = {key.strip(): value for key, value in last_row.items()}
        last_row["model"] = model_label_from_run(run_name)
        last_row["run"] = run_name
        rows.append(last_row)

    return pd.DataFrame(rows)


model_comparison = collect_training_metrics(TRAINING_RUNS_DIR)
model_comparison


In [ ]:
def plot_metric_bars(metrics_df):
    if metrics_df.empty:
        print("Brak zapisanych wynik?w treningu. Najpierw uruchom kom?rk? z RUN_TRAINING = True.")
        return

    metric_candidates = [
        "metrics/accuracy_top1",
        "metrics/accuracy_top5",
        "val/loss",
        "train/loss",
    ]
    metric_columns = [column for column in metric_candidates if column in metrics_df.columns]

    if not metric_columns:
        print("Nie znaleziono standardowych kolumn metryk w results.csv.")
        display(metrics_df)
        return

    fig, axes = plt.subplots(1, len(metric_columns), figsize=(5 * len(metric_columns), 4))
    if len(metric_columns) == 1:
        axes = [axes]

    for ax, metric in zip(axes, metric_columns):
        ax.bar(metrics_df["model"], metrics_df[metric])
        ax.set_title(metric)
        ax.set_xlabel("Model")
        ax.tick_params(axis="x", rotation=30)
        ax.grid(axis="y", alpha=0.25)

    plt.tight_layout()
    plt.show()


plot_metric_bars(model_comparison)


In [ ]:
def plot_training_curves(runs_dir):
    if not runs_dir.exists():
        print("Brak katalogu runs z wynikami treningu.")
        return

    curves = []
    for results_csv in sorted(runs_dir.glob("*/results.csv")):
        run_name = results_csv.parent.name
        metrics = pd.read_csv(results_csv)
        metrics.columns = [column.strip() for column in metrics.columns]
        metrics["model"] = model_label_from_run(run_name)
        curves.append(metrics)

    if not curves:
        print("Brak plik?w results.csv. Najpierw wykonaj trening.")
        return

    metric_candidates = ["metrics/accuracy_top1", "metrics/accuracy_top5", "val/loss", "train/loss"]
    available_metrics = [metric for metric in metric_candidates if any(metric in curve.columns for curve in curves)]

    fig, axes = plt.subplots(1, len(available_metrics), figsize=(5 * len(available_metrics), 4))
    if len(available_metrics) == 1:
        axes = [axes]

    for ax, metric in zip(axes, available_metrics):
        for curve in curves:
            if metric not in curve.columns:
                continue
            x_values = curve["epoch"] if "epoch" in curve.columns else range(1, len(curve) + 1)
            ax.plot(x_values, curve[metric], marker="o", label=curve["model"].iloc[0])

        ax.set_title(metric)
        ax.set_xlabel("Epoka")
        ax.grid(alpha=0.25)
        ax.legend()

    plt.tight_layout()
    plt.show()


plot_training_curves(TRAINING_RUNS_DIR)


# Rozpoznawanie znak?w na przyk?adach

Ta cz??? wykonuje predykcj? na kilku przyk?adowych obrazach i pokazuje wyniki bezpo?rednio pod zdj?ciami. Je?eli dost?pnych jest kilka modeli, ka?dy obraz dostanie osobny wynik dla ka?dego wariantu YOLO11.


In [ ]:
def choose_prediction_examples(class_dirs, examples_per_class=1, max_classes=8):
    selected_images = []

    for class_dir in class_dirs[:max_classes]:
        image_paths = sorted(
            path for path in class_dir.rglob("*")
            if path.suffix.lower() in image_extensions
        )
        selected_images.extend(image_paths[:examples_per_class])

    return selected_images


example_images = choose_prediction_examples(
    class_dirs=class_dirs,
    examples_per_class=1,
    max_classes=8,
)

example_images


In [ ]:
def predict_with_trained_models(model_paths, image_paths):
    if not model_paths:
        print("Brak wytrenowanego modelu. Najpierw uruchom trening albo upewnij si?, ?e istnieje plik models/yolo11n-best.pt.")
        return pd.DataFrame()

    prediction_rows = []

    for run_name, model_path in model_paths.items():
        model = YOLO(str(model_path))

        for image_path in image_paths:
            result = model.predict(str(image_path), verbose=False)[0]
            class_id = int(result.probs.top1)
            confidence = float(result.probs.top1conf)

            prediction_rows.append({
                "model": model_label_from_run(run_name),
                "image_path": image_path,
                "true_class": image_path.parent.name,
                "predicted_class": result.names[class_id],
                "confidence": confidence,
                "is_correct": image_path.parent.name == result.names[class_id],
            })

    return pd.DataFrame(prediction_rows)


prediction_results = predict_with_trained_models(trained_model_paths, example_images)
prediction_results


In [ ]:
def show_prediction_grid(image_paths, prediction_results, columns=4):
    if prediction_results.empty:
        print("Brak wynik?w predykcji do wy?wietlenia.")
        return

    rows = math.ceil(len(image_paths) / columns)
    fig, axes = plt.subplots(rows, columns, figsize=(4.2 * columns, 4.8 * rows))
    axes = axes.flatten() if hasattr(axes, "flatten") else [axes]

    for ax, image_path in zip(axes, image_paths):
        image_predictions = prediction_results[prediction_results["image_path"] == image_path]
        image = plt.imread(image_path)
        ax.imshow(image)
        ax.axis("off")

        prediction_lines = []
        for row in image_predictions.itertuples(index=False):
            mark = "OK" if row.is_correct else "ERR"
            prediction_lines.append(
                f"{row.model}: {row.predicted_class} ({row.confidence:.2f}) {mark}"
            )

        title = f"Real: {image_path.parent.name}\n" + "\n".join(prediction_lines)
        ax.set_title(title, fontsize=9)

    for ax in axes[len(image_paths):]:
        ax.axis("off")

    plt.tight_layout()
    plt.show()


show_prediction_grid(example_images, prediction_results)


In [ ]:
if not prediction_results.empty:
    recognition_summary = (
        prediction_results
        .groupby("model")
        .agg(
            examples=("image_path", "count"),
            correct=("is_correct", "sum"),
            mean_confidence=("confidence", "mean"),
        )
        .reset_index()
    )
    recognition_summary["sample_accuracy"] = recognition_summary["correct"] / recognition_summary["examples"]
else:
    recognition_summary = pd.DataFrame()

recognition_summary


# Wnioski

Notebook zawiera pe?ny pipeline w jednym pliku: analiz? danych, podzia? na `train/val/test`, przygotowanie formatu YOLO11, trening, por?wnanie wariant?w YOLO11, zapis najlepszego modelu oraz wizualne rozpoznawanie znak?w na przyk?adach. Domy?lnie d?ugie kroki treningowe s? sterowane flagami, wi?c mo?na najpierw uruchomi? analiz?, a potem w??czy? trening tylko dla wybranego modelu.
